# Langfuse v3 — Dataset & Run Explorer / Exporter (REST · Judge · Circulars · Date-Filter)

Explore datasets, runs, run items, **LLM-as-a-judge scores + reasoning + judge input prompt**,
**circular metadata** from the retriever step, and **filter circulars to only those whose date
the answer actually cites** (Arabic + numeric dates, two-pass strict→loose). Export to a styled
read-ready Excel, a slim answers-only Excel, and flat + nested JSON.

Talks to the Langfuse **Public REST API** directly via `httpx` (no SDK, no OTel) so self-signed /
TLS-proxy SSL issues don't apply.

---
### Extraction map
- **Dataset / item / run** metadata
- **Run items** + main trace: app `input`, `actual_output`, dataset `expected_output`
- **Judge**: score `value`, `reasoning` (score.comment), judge `input` prompt + raw `output`
- **Circulars**: from obs *"Vector Store Retriever (detailed)"* → `output.documents[*]`
- **Date filter**: parse dates cited in `actual_output` (Arabic month names both systems +
  Arabic-Indic digits + numeric), keep only matching circulars. Strict day-match first, loose
  month-match as fallback.

## 1. Connection (REST client)

In [ ]:
import httpx

LANGFUSE_HOST       = "https://your-langfuse.internal"   # no trailing slash
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

_client = httpx.Client(
    base_url=LANGFUSE_HOST,
    auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
    verify=False,
    trust_env=False,
    timeout=httpx.Timeout(30.0, read=120.0),
)

def lf_get(path, **params):
    r = _client.get(path, params={k: v for k, v in params.items() if v is not None})
    r.raise_for_status()
    return r.json()

h = _client.get("/api/public/health"); h.raise_for_status()
print("Connected OK ->", LANGFUSE_HOST, "|", h.json())

## 2. Configuration — control panel

`FIELD_DEFINITIONS` doubles as README documentation: `(include?, "definition")`.

In [ ]:
# ============================================================
#  TRACE / RUN-ITEM FIELDS  — (include?, README definition)
# ============================================================
FIELD_DEFINITIONS = {
    "run_name":        (True,  "Name of the experiment/run this row belongs to."),
    "dataset_item_id": (True,  "UUID of the dataset test case that was executed."),
    "input":           (True,  "What the application actually received (main trace input)."),
    "expected_output": (True,  "Gold/reference answer from the dataset item (ground truth)."),
    "actual_output":   (True,  "What the application actually produced (main trace output)."),
    "extracted_metadata": (True, "Circulars kept after date-filtering: timestamp | filename | attachment_id | summary, one per line."),
    "cited_dates":     (True,  "Dates parsed from the answer text (Arabic/numeric) used to filter circulars."),
    "match_tier":      (True,  "Which matching pass kept the circulars: strict (exact day), loose (month), no-dates (kept all), or none."),
    "reasoning_chain": (False, "Concatenated outputs of the app's own intermediate steps (NOT the judge)."),
    "n_observations":  (False, "Count of application-side observations on the main trace."),
    "trace_name":      (False, "Name of the main trace."),
    "latency_ms":      (False, "End-to-end latency of the main trace, milliseconds."),
    "total_cost":      (False, "Total model cost recorded on the main trace."),
    "trace_id":        (True,  "UUID of the main trace (cross-reference in the UI)."),
    "run_item_id":     (False, "UUID linking the dataset item to this run execution."),
    "observation_id":  (False, "Specific observation id, if the run item is scoped to one."),
}
FIELDS = {k: v[0] for k, v in FIELD_DEFINITIONS.items()}

# ============================================================
#  JUDGE / EVALUATOR
# ============================================================
JUDGE_INCLUDE         = True
JUDGE_FETCH_REASONING = True
JUDGE_FETCH_PROMPT    = True
JUDGE_DEFINITIONS = {
    "score":          (True,  "The evaluator's score value (numeric/boolean) or category."),
    "reasoning":      (JUDGE_FETCH_REASONING, "The judge's written justification (score.comment)."),
    "judge_prompt":   (JUDGE_FETCH_PROMPT,    "The exact prompt the judge LLM received (judge execution-trace input)."),
    "judge_response": (JUDGE_FETCH_PROMPT,    "The judge LLM's raw response (judge execution-trace output)."),
}

# ============================================================
#  CIRCULAR METADATA
# ============================================================
EXTRACT_CIRCULAR_METADATA = True
RETRIEVER_OBS_NAME = "Vector Store Retriever (detailed)"  # exact name (case-insensitive match in code)
RETRIEVER_OBS_TYPE = "retriever"   # require this type; set None to match on name only
CIRCULAR_DOCS_KEY  = "documents"
CIRCULAR_FIELDS    = ["timestamp", "filename", "attachment_id", "summary"]
# extra fields available in your docs if you want them:
# score, doc_name, page_image_filename, slice_image_filename, page_idx, slice_idx, img_url, query

# ============================================================
#  CIRCULAR DATE-FILTERING (two-pass strict -> loose)
# ============================================================
FILTER_CIRCULARS_BY_ANSWER_DATES = True   # keep only circulars whose date is cited in actual_output
STRICT_MATCH_MODE = "day"                 # first pass (precise): "day"
LOOSE_MATCH_MODE  = "month"               # fallback if strict finds nothing: "month" (or "year")
ENABLE_LOOSE_FALLBACK = True              # if False, only the strict pass runs
KEEP_UNMATCHED_IF_NO_DATES = True         # answer cites NO parseable date -> keep all (True) or none (False)

# ============================================================
#  GENERAL
# ============================================================
TEXT_LIMIT = 8000

INCLUDE_APP_OBSERVATIONS = bool(FIELDS.get("reasoning_chain") or FIELDS.get("n_observations"))
_NEED_OBS = INCLUDE_APP_OBSERVATIONS or EXTRACT_CIRCULAR_METADATA

print("Active fields :", [k for k, v in FIELDS.items() if v])
print("Date filter   :", FILTER_CIRCULARS_BY_ANSWER_DATES,
      f"(strict={STRICT_MATCH_MODE}, loose={LOOSE_MATCH_MODE if ENABLE_LOOSE_FALLBACK else 'off'})")
print("Fetch app obs :", _NEED_OBS)

## 3. Imports & helpers

In [ ]:
import json, datetime as dt
from pathlib import Path
from urllib.parse import quote
import pandas as pd

OUT_DIR = Path("langfuse_exports"); OUT_DIR.mkdir(exist_ok=True)
def _ts(): return dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def _flatten(value, max_len=None):
    if value is None: return None
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, default=str)
    value = str(value)
    if max_len and len(value) > max_len:
        value = value[:max_len] + " ...[truncated]"
    return value

def _as_obj(v):
    if v is None: return None
    if isinstance(v, (dict, list)): return v
    if isinstance(v, str):
        try: return json.loads(v)
        except Exception: return None
    return None

def paginate(path, data_key="data", **params):
    page, out = 1, []
    while True:
        js = lf_get(path, page=page, limit=100, **params)
        data = js.get(data_key, []) or []
        if not data: break
        out.extend(data)
        meta = js.get("meta") or {}
        tp = meta.get("totalPages")
        if tp is not None and page >= tp: break
        if tp is None and len(data) < 100: break
        page += 1
    return out

def _apply_fields(row):
    return {k: row.get(k) for k, v in FIELDS.items() if v}

print("Helpers ready. Exports ->", OUT_DIR.resolve())

## 3b. Arabic/numeric date parsing + two-pass circular filter

Parses dates cited in the answer (Arabic month names — both MSA/Gulf and Levantine/Syriac —
plus Arabic-Indic digits and numeric formats), and matches them to circular `timestamp`s.

**Two-pass:** strict (exact day) first; if that keeps nothing for an answer, fall back to
loose (month). Per-row `match_tier` records which pass succeeded.

In [ ]:
import re

_AR_MONTHS = {
    # MSA / Gulf / Egypt
    "يناير":1,"فبراير":2,"مارس":3,"أبريل":4,"ابريل":4,"مايو":5,"يونيو":6,"يونية":6,
    "يوليو":7,"يولية":7,"أغسطس":8,"اغسطس":8,"سبتمبر":9,"أكتوبر":10,"اكتوبر":10,
    "نوفمبر":11,"ديسمبر":12,
    # Levantine / Syriac / Iraqi
    "كانون الثاني":1,"شباط":2,"آذار":3,"اذار":3,"نيسان":4,"أيار":5,"ايار":5,
    "حزيران":6,"تموز":7,"آب":8,"اب":8,"أيلول":9,"ايلول":9,
    "تشرين الأول":10,"تشرين الاول":10,"تشرين الثاني":11,"كانون الأول":12,"كانون الاول":12,
}
_AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩،", "0123456789,")
def _norm_digits(s): return s.translate(_AR_DIGITS) if s else s
_MONTH_ALT = "|".join(sorted((re.escape(m) for m in _AR_MONTHS), key=len, reverse=True))
_RE_AR  = re.compile(r"(?:(\d{1,2})\s+)?(" + _MONTH_ALT + r")\s+(\d{4})")
_RE_ISO = re.compile(r"\b(\d{4})[-/](\d{1,2})[-/](\d{1,2})\b")
_RE_DMY = re.compile(r"\b(\d{1,2})[-/](\d{1,2})[-/](\d{4})\b")

def parse_answer_dates(text):
    """All dates cited in the answer -> set of (year, month, day|None)."""
    if not text: return set()
    t = _norm_digits(str(text)); found = set()
    for m in _RE_AR.finditer(t):          # arabic month names (both systems), day optional
        d, mn, y = m.groups(); mo = _AR_MONTHS.get(mn)
        if mo: found.add((int(y), mo, int(d) if d else None))
    for m in _RE_ISO.finditer(t):         # numeric YYYY-MM-DD
        y, mo, d = map(int, m.groups()); found.add((y, mo, d))
    for m in _RE_DMY.finditer(t):         # numeric DD/MM/YYYY
        d, mo, y = map(int, m.groups()); found.add((y, mo, d))
    return found

def parse_circular_date(ts):
    """Circular timestamp -> (year, month, day) with format fallbacks."""
    if not ts: return None
    s = str(ts).strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S%z","%Y-%m-%dT%H:%M:%SZ","%Y-%m-%dT%H:%M:%S",
                "%Y-%m-%d %H:%M:%S","%Y-%m-%d"):
        try:
            ss = s.replace("Z","+0000") if (fmt.endswith("%z") and s.endswith("Z")) else s
            d = dt.datetime.strptime(ss, fmt); return (d.year, d.month, d.day)
        except Exception: pass
    m = re.search(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})", s)   # last-resort scan
    if m: y, mo, d = map(int, m.groups()); return (y, mo, d)
    return None

def _match_one(cited, tt, mode):
    if tt is None: return False
    ty, tm, td = tt
    for (ay, am, ad) in cited:
        if mode == "year"  and ay == ty: return True
        if mode == "month" and (ay, am) == (ty, tm): return True
        if mode == "day" and ad is not None and (ay, am, ad) == (ty, tm, td): return True
    return False

def filter_circulars_two_pass(circs, actual_output):
    """Return (kept, cited_dates_set, tier). Honors cell-2 toggles.
       tier in {strict, loose, no-dates, none}."""
    cited = parse_answer_dates(actual_output)
    if not cited:
        return (circs if KEEP_UNMATCHED_IF_NO_DATES else []), cited, "no-dates"
    strict = [c for c in circs
              if _match_one(cited, parse_circular_date(c.get("timestamp")), STRICT_MATCH_MODE)]
    if strict:
        return strict, cited, "strict"
    if ENABLE_LOOSE_FALLBACK:
        loose = [c for c in circs
                 if _match_one(cited, parse_circular_date(c.get("timestamp")), LOOSE_MATCH_MODE)]
        if loose:
            return loose, cited, "loose"
    return [], cited, "none"

def _fmt_cited(cited):
    def one(t):
        y, m, d = t
        return f"{y}-{m:02d}" + (f"-{d:02d}" if d else "")
    return ", ".join(one(t) for t in sorted(cited)) if cited else None

print("Two-pass Arabic/numeric date filter ready.")

## 4. List all datasets

In [ ]:
datasets = paginate("/api/public/v2/datasets")
df_datasets = pd.DataFrame([{
    "name": d.get("name"), "id": d.get("id"),
    "description": d.get("description"),
    "metadata": _flatten(d.get("metadata")),
    "createdAt": d.get("createdAt"), "updatedAt": d.get("updatedAt"),
} for d in datasets])
print(f"{len(df_datasets)} dataset(s)\n")
df_datasets

## 5. Pick a dataset

In [ ]:
DATASET_NAME = "your-dataset-name"   # <-- edit me
dataset = lf_get(f"/api/public/v2/datasets/{quote(DATASET_NAME, safe='')}")
print("Dataset:", dataset.get("name"), "| id:", dataset.get("id"))

## 6. Mode A — Dataset items only
Filtered by `datasetName` (the `datasetId` filter is silently ignored upstream).

In [ ]:
items = paginate("/api/public/dataset-items", datasetName=DATASET_NAME)
df_items = pd.DataFrame([{
    "item_id": it.get("id"), "status": it.get("status"),
    "input": _flatten(it.get("input")),
    "expected_output": _flatten(it.get("expectedOutput")),
    "metadata": _flatten(it.get("metadata")),
    "source_trace_id": it.get("sourceTraceId"),
    "createdAt": it.get("createdAt"),
} for it in items])
print(f"{len(df_items)} item(s)")
df_items.head(20)

## 7. List runs for this dataset

In [ ]:
runs = paginate(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs")
df_runs = pd.DataFrame([{
    "run_name": r.get("name"), "run_id": r.get("id"),
    "description": r.get("description"),
    "metadata": _flatten(r.get("metadata")),
    "createdAt": r.get("createdAt"), "updatedAt": r.get("updatedAt"),
} for r in runs])
print(f"{len(df_runs)} run(s)")
df_runs

## 8. Mode B — Runs + outputs + judge + circulars (date-filtered)

Per run item: main trace (input/actual), dataset item (expected), judge scores, circulars
from the retriever step **filtered to dates cited in the answer** (two-pass).

In [ ]:
SELECTED_RUNS = []   # [] = all runs, or e.g. ["baseline-v1"]

target = df_runs["run_name"].tolist() if not SELECTED_RUNS else SELECTED_RUNS
print(f"Pulling {len(target)} run(s)...")

_trace_cache, _obs_cache, _item_cache = {}, {}, {}

def get_run_with_items(run_name):
    return lf_get(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs/{quote(run_name, safe='')}")

def get_trace(trace_id):
    if not trace_id: return None
    if trace_id not in _trace_cache:
        try: _trace_cache[trace_id] = lf_get(f"/api/public/traces/{trace_id}")
        except Exception as e:
            _trace_cache[trace_id] = None; print(f"  ! trace {trace_id}: {e}")
    return _trace_cache[trace_id]

def get_dataset_item(item_id):
    if not item_id: return None
    if item_id not in _item_cache:
        try: _item_cache[item_id] = lf_get(f"/api/public/dataset-items/{item_id}")
        except Exception: _item_cache[item_id] = None
    return _item_cache[item_id]

def get_app_observations(trace_id):
    if not trace_id: return []
    if trace_id not in _obs_cache:
        try: _obs_cache[trace_id] = paginate("/api/public/observations", traceId=trace_id)
        except Exception: _obs_cache[trace_id] = []
    return _obs_cache[trace_id]

def _score_execution_trace_id(score):
    for k in ("executionTraceId", "evalExecutionTraceId", "jobExecutionId", "executionId"):
        if score.get(k): return score.get(k)
    md_ = score.get("metadata") or {}
    if isinstance(md_, dict):
        for k in ("executionTraceId", "traceId", "execution_trace_id"):
            if md_.get(k): return md_.get(k)
    return None

def build_judge_detail(score):
    d = {"name": score.get("name"), "value": score.get("value"),
         "string_value": score.get("stringValue"), "data_type": score.get("dataType"),
         "reasoning": score.get("comment") if JUDGE_FETCH_REASONING else None,
         "judge_prompt": None, "judge_response": None, "execution_trace_id": None}
    if JUDGE_FETCH_PROMPT:
        exec_id = _score_execution_trace_id(score)
        d["execution_trace_id"] = exec_id
        if exec_id:
            et = get_trace(exec_id)
            if et:
                d["judge_prompt"]   = et.get("input")
                d["judge_response"] = et.get("output")
    return d

def extract_circulars(observations):
    """retriever obs -> output.documents[*] -> CIRCULAR_FIELDS (case-insensitive name match)."""
    found = []
    target_name = RETRIEVER_OBS_NAME.strip().lower()
    for o in observations:
        if (o.get("name") or "").strip().lower() != target_name:
            continue
        if RETRIEVER_OBS_TYPE and (o.get("type") or "").lower() != RETRIEVER_OBS_TYPE.lower():
            continue
        out = _as_obj(o.get("output"))
        docs = (out or {}).get(CIRCULAR_DOCS_KEY) if isinstance(out, dict) else None
        if not isinstance(docs, list):
            continue
        for doc in docs:
            if isinstance(doc, dict):
                found.append({f: doc.get(f) for f in CIRCULAR_FIELDS})
    return found

def circulars_to_cell(circs):
    if not circs: return None
    return "\n".join(" | ".join("" if c.get(f) is None else str(c.get(f)) for f in CIRCULAR_FIELDS)
                      for c in circs)

flat_rows, nested = [], []
tier_counts = {}

for run_name in target:
    run_full = get_run_with_items(run_name)
    run_items = run_full.get("datasetRunItems") or run_full.get("data") or []
    print(f"  run '{run_name}': {len(run_items)} item(s)")
    run_node = {"run": {k: v for k, v in run_full.items()
                        if k not in ("datasetRunItems", "data")}, "items": []}

    for it in run_items:
        trace_id = it.get("traceId")
        item_id  = it.get("datasetItemId")
        di = get_dataset_item(item_id)
        expected = di.get("expectedOutput") if di else None
        trace = get_trace(trace_id)
        actual_out_raw = trace.get("output") if trace else None

        observations = get_app_observations(trace_id) if _NEED_OBS else []

        reasoning = None
        if FIELDS.get("reasoning_chain"):
            parts = [f"[{o.get('type','')}:{o.get('name','')}] {_flatten(o.get('output'),2000)}"
                     for o in observations if o.get("output") is not None]
            reasoning = "\n".join(parts) if parts else None

        # circulars: extract all, then date-filter
        circs_all = extract_circulars(observations) if EXTRACT_CIRCULAR_METADATA else []
        if EXTRACT_CIRCULAR_METADATA and FILTER_CIRCULARS_BY_ANSWER_DATES:
            circs, cited_dates, tier = filter_circulars_two_pass(circs_all, actual_out_raw)
        else:
            circs, cited_dates, tier = circs_all, set(), ("disabled" if EXTRACT_CIRCULAR_METADATA else "n/a")
        tier_counts[tier] = tier_counts.get(tier, 0) + 1

        scores = (trace.get("scores") if trace else None) or []
        judge_details = [build_judge_detail(s) for s in scores] if JUDGE_INCLUDE else []

        base = {
            "run_name": run_name,
            "dataset_item_id": item_id,
            "input": _flatten(trace.get("input") if trace else None, TEXT_LIMIT),
            "expected_output": _flatten(expected, TEXT_LIMIT),
            "actual_output": _flatten(actual_out_raw, TEXT_LIMIT),
            "extracted_metadata": circulars_to_cell(circs),
            "cited_dates": _fmt_cited(cited_dates),
            "match_tier": tier,
            "reasoning_chain": _flatten(reasoning, TEXT_LIMIT),
            "n_observations": len(observations) if _NEED_OBS else None,
            "trace_name": trace.get("name") if trace else None,
            "latency_ms": trace.get("latency") if trace else None,
            "total_cost": trace.get("totalCost") if trace else None,
            "trace_id": trace_id,
            "run_item_id": it.get("id"),
            "observation_id": it.get("observationId"),
        }
        row = _apply_fields(base)

        if JUDGE_INCLUDE:
            for jd in judge_details:
                nm = jd["name"] or "score"
                suffix = "" if len(judge_details) == 1 else f"__{nm}"
                val = jd["value"] if jd["value"] is not None else jd["string_value"]
                if JUDGE_DEFINITIONS["score"][0]:
                    row[f"score{suffix}"] = val
                if JUDGE_DEFINITIONS["reasoning"][0]:
                    row[f"reasoning{suffix}"] = _flatten(jd["reasoning"], TEXT_LIMIT)
                if JUDGE_DEFINITIONS["judge_prompt"][0]:
                    row[f"judge_prompt{suffix}"]   = _flatten(jd["judge_prompt"], TEXT_LIMIT)
                    row[f"judge_response{suffix}"] = _flatten(jd["judge_response"], TEXT_LIMIT)

        flat_rows.append(row)
        run_node["items"].append({
            "run_item": it, "dataset_item": di, "trace": trace,
            "observations": observations,
            "circulars": circs,            # date-filtered (kept)
            "circulars_all": circs_all,    # full retriever set (archive)
            "cited_dates": _fmt_cited(cited_dates), "match_tier": tier,
            "scores": scores, "judge_details": judge_details,
        })
    nested.append(run_node)

df_run_items = pd.DataFrame(flat_rows)
print(f"\nTotal run items: {len(df_run_items)}")
print("Match tiers   :", tier_counts)
print("Columns       :", list(df_run_items.columns))
df_run_items.head(20)

## 8b. Circular-extraction diagnostic + autosuggest
Reads cached `nested`. If `extracted_metadata` is empty, tells you why and suggests config.

In [ ]:
from collections import Counter

def _arr_keys(d):
    out = []
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, list) and v and isinstance(v[0], dict): out.append(k)
    return out

rows = []; all_names = Counter(); retr_like = Counter(); doc_keys = Counter(); doc_fields = Counter()
for run_node in nested:
    rn = (run_node.get("run") or {}).get("name")
    for item in run_node["items"]:
        obs = item.get("observations") or []
        circs_all = item.get("circulars_all") or []
        for o in obs: all_names[o.get("name")] += 1
        like = [o for o in obs if "retriev" in (o.get("type") or "").lower()
                or "retriev" in (o.get("name") or "").lower()]
        for o in like:
            retr_like[o.get("name")] += 1
            ok = _as_obj(o.get("output"))
            for k in _arr_keys(ok if isinstance(ok, dict) else {}):
                doc_keys[k] += 1
                if ok[k] and isinstance(ok[k][0], dict):
                    for f in ok[k][0].keys(): doc_fields[f] += 1
        if not obs: verdict = "no-observations"
        elif circs_all: verdict = "extracted-ok"
        elif not like: verdict = "name-mismatch"
        else:
            out = _as_obj(like[0].get("output"))
            verdict = ("no-output" if out is None
                       else "key-ok-but-empty" if (isinstance(out, dict) and CIRCULAR_DOCS_KEY in out)
                       else "key-mismatch")
        rows.append({"run_name": rn, "n_obs": len(obs),
                     "n_extracted": len(circs_all), "verdict": verdict})

diag = pd.DataFrame(rows)
print(diag["verdict"].value_counts().to_string(), "\n")
print("OBSERVATION NAMES (top 10):")
for n, c in all_names.most_common(10):
    print(f"  {c:>4}x  {n!r}{'  <-- configured' if n == RETRIEVER_OBS_NAME else ''}")
if (diag["verdict"] != "extracted-ok").any():
    print("\nAUTOSUGGEST:")
    if retr_like:
        bn = retr_like.most_common(1)[0][0]
        print(f"  RETRIEVER_OBS_NAME = {bn!r}" + ("  (matches)" if bn == RETRIEVER_OBS_NAME else "  <-- update"))
    if doc_keys:
        bk = doc_keys.most_common(1)[0][0]
        print(f"  CIRCULAR_DOCS_KEY  = {bk!r}" + ("  (matches)" if bk == CIRCULAR_DOCS_KEY else "  <-- update"))
    if doc_fields:
        print(f"  fields in docs     : {[f for f, _ in doc_fields.most_common()]}")
diag.head(20)

## 9. Inspect one full record

In [ ]:
if not df_run_items.empty:
    for k, v in df_run_items.iloc[0].to_dict().items():
        s = str(v)
        print(f"--- {k} ---\n{s[:1500]}{' ...[more]' if len(s) > 1500 else ''}\n")
else:
    print("No run items - run cell 8 first.")

## 10. Export — styled Excel + slim Excel + flat JSON + nested JSON

- **Excel** (full): README + datasets / items / runs / **run_items** (ordered, styled).
- **Excel** (slim): `query`, `actual_output`, `extracted_metadata` only.
- **Flat JSON**: selected fields + config header.
- **Nested JSON**: raw archive — full traces, all scores, **both** filtered `circulars` and
  full `circulars_all`. Ignores field toggles.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

EXPORT_DATASETS, EXPORT_ITEMS, EXPORT_RUNS, EXPORT_RUN_ITEMS, EXPORT_NESTED_JSON = True, True, True, True, True
EXPORT_SLIM_EXCEL = True

stamp = _ts(); safe = quote(DATASET_NAME, safe="")

def ordered_run_item_columns(df):
    lead = [c for c in ["run_name", "dataset_item_id", "input", "expected_output",
                        "actual_output", "extracted_metadata", "cited_dates", "match_tier"] if c in df.columns]
    judge = [c for c in df.columns if c.startswith(("score", "reasoning", "judge_prompt", "judge_response"))
             and c != "reasoning_chain"]
    tail = [c for c in ["reasoning_chain", "n_observations", "trace_name", "latency_ms",
                        "total_cost", "trace_id", "run_item_id", "observation_id"] if c in df.columns]
    seen = set(lead + judge + tail)
    return lead + judge + [c for c in df.columns if c not in seen] + tail

WIDE_WRAP = {"input", "expected_output", "actual_output", "extracted_metadata",
             "reasoning", "reasoning_chain", "judge_prompt", "judge_response", "query"}
NARROW    = {"run_name", "dataset_item_id", "trace_id", "run_item_id", "observation_id",
             "score", "latency_ms", "total_cost", "n_observations", "match_tier", "cited_dates"}

HEADER_FILL = PatternFill("solid", fgColor="2F5597")
HEADER_FONT = Font(bold=True, color="FFFFFF", size=11)
THIN = Side(style="thin", color="D9D9D9")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
TOPLEFT = Alignment(horizontal="left", vertical="top", wrap_text=True)

def style_sheet(ws, wrap_cols=None, freeze=True):
    wrap_cols = wrap_cols or set()
    for cell in ws[1]:
        cell.fill = HEADER_FILL; cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True); cell.border = BORDER
    headers = [c.value for c in ws[1]]
    for idx, name in enumerate(headers, start=1):
        col = get_column_letter(idx)
        if name in wrap_cols: w = 60
        elif name in NARROW: w = 18
        elif str(name).startswith(("judge_prompt", "judge_response", "reasoning")): w = 60
        else: w = 28
        ws.column_dimensions[col].width = w
    for r in range(2, ws.max_row + 1):
        for idx, name in enumerate(headers, start=1):
            cell = ws.cell(row=r, column=idx); cell.border = BORDER
            cell.alignment = TOPLEFT if name in wrap_cols else Alignment(horizontal="left", vertical="top", wrap_text=False)
    if ws.max_row >= 1:
        ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    if freeze: ws.freeze_panes = "A2"

def build_readme_df():
    rows = [("SHEET: run_items", "")]
    for k, (inc, desc) in FIELD_DEFINITIONS.items():
        if inc: rows.append((k, desc))
    for k, (inc, desc) in JUDGE_DEFINITIONS.items():
        if inc: rows.append((k + " (+__<evaluator> if >1)", desc))
    rows += [("", ""),
             ("extracted_metadata format", "Each line: " + " | ".join(CIRCULAR_FIELDS)),
             ("date filter", f"strict={STRICT_MATCH_MODE}, loose={LOOSE_MATCH_MODE if ENABLE_LOOSE_FALLBACK else 'off'}; circulars kept only if their date is cited in the answer"),
             ("source observation", f'name="{RETRIEVER_OBS_NAME}", type="{RETRIEVER_OBS_TYPE}", output.{CIRCULAR_DOCS_KEY}[*]'),
             ("", ""), ("OTHER SHEETS", ""),
             ("datasets", "All datasets in the project."),
             ("items", "Dataset test cases (Mode A)."),
             ("runs", "Experiments/runs for the selected dataset.")]
    return pd.DataFrame(rows, columns=["Field / Sheet", "Definition"])

# ---- full Excel ----
xlsx_path = OUT_DIR / f"{safe}_{stamp}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xl:
    build_readme_df().to_excel(xl, sheet_name="README", index=False)
    if EXPORT_DATASETS and not df_datasets.empty:
        df_datasets.to_excel(xl, sheet_name="datasets", index=False)
    if EXPORT_ITEMS and 'df_items' in dir() and not df_items.empty:
        df_items.to_excel(xl, sheet_name="items", index=False)
    if EXPORT_RUNS and 'df_runs' in dir() and not df_runs.empty:
        df_runs.to_excel(xl, sheet_name="runs", index=False)
    if EXPORT_RUN_ITEMS and 'df_run_items' in dir() and not df_run_items.empty:
        df_run_items[ordered_run_item_columns(df_run_items)].to_excel(xl, sheet_name="run_items", index=False)
    wb = xl.book
    rm = wb["README"]
    for cell in rm[1]: cell.fill = HEADER_FILL; cell.font = HEADER_FONT
    rm.column_dimensions["A"].width = 44; rm.column_dimensions["B"].width = 95
    for r in range(2, rm.max_row + 1):
        rm.cell(row=r, column=1).font = Font(bold=True)
        rm.cell(row=r, column=2).alignment = Alignment(wrap_text=True, vertical="top")
    rm.freeze_panes = "A2"
    for name in ("datasets", "items", "runs"):
        if name in wb.sheetnames: style_sheet(wb[name], wrap_cols={"metadata", "description", "input", "expected_output"})
    if "run_items" in wb.sheetnames: style_sheet(wb["run_items"], wrap_cols=WIDE_WRAP)
print("Excel (full) ->", xlsx_path)

# ---- slim Excel: query / actual_output / extracted_metadata ----
if EXPORT_SLIM_EXCEL and EXPORT_RUN_ITEMS and 'df_run_items' in dir() and not df_run_items.empty:
    rename = {"input": "query", "actual_output": "actual_output", "extracted_metadata": "extracted_metadata"}
    cols = [c for c in rename if c in df_run_items.columns]
    df_slim = df_run_items[cols].rename(columns=rename)
    slim_path = OUT_DIR / f"{safe}_{stamp}_slim.xlsx"
    with pd.ExcelWriter(slim_path, engine="openpyxl") as xl:
        df_slim.to_excel(xl, sheet_name="answers", index=False)
        style_sheet(xl.book["answers"], wrap_cols={"query", "actual_output", "extracted_metadata"})
    print("Excel (slim) ->", slim_path)

# ---- flat JSON ----
payload = {"exported_at": stamp, "host": LANGFUSE_HOST, "dataset_name": DATASET_NAME,
           "field_selection": FIELDS,
           "judge_config": {k: v[0] for k, v in JUDGE_DEFINITIONS.items()},
           "circular_config": {"enabled": EXTRACT_CIRCULAR_METADATA, "obs_name": RETRIEVER_OBS_NAME,
                               "fields": CIRCULAR_FIELDS},
           "date_filter": {"enabled": FILTER_CIRCULARS_BY_ANSWER_DATES, "strict": STRICT_MATCH_MODE,
                           "loose": LOOSE_MATCH_MODE if ENABLE_LOOSE_FALLBACK else None,
                           "keep_if_no_dates": KEEP_UNMATCHED_IF_NO_DATES}}
if EXPORT_ITEMS and 'df_items' in dir():    payload["items"] = df_items.to_dict(orient="records")
if EXPORT_RUNS and 'df_runs' in dir():       payload["runs"] = df_runs.to_dict(orient="records")
if EXPORT_RUN_ITEMS and 'df_run_items' in dir(): payload["run_items_flat"] = df_run_items.to_dict(orient="records")
json_path = OUT_DIR / f"{safe}_{stamp}.json"
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str))
print("JSON (flat)  ->", json_path)

# ---- nested JSON (raw archive) ----
if EXPORT_NESTED_JSON and 'nested' in dir() and nested:
    nested_path = OUT_DIR / f"{safe}_{stamp}_nested.json"
    nested_path.write_text(json.dumps(nested, ensure_ascii=False, indent=2, default=str))
    print("JSON (nested)->", nested_path)
print("\nDone.")

---
### Notes
- **Date filter** keeps circulars whose `timestamp` matches a date cited in `actual_output`.
  Strict (exact day) runs first; if it keeps nothing for an answer, loose (month) is the
  fallback. `match_tier` per row tells you which pass applied (`strict` / `loose` / `no-dates` / `none`).
- **Arabic dates**: both month systems (أغسطس … / آب …), Arabic-Indic digits (٣١), month-only
  citations, and numeric (2025-08-31, 31/08/2025) all parse. If a real answer uses a format that
  comes back unparsed (`match_tier=none` where you expect a match), paste one example to extend.
- **Both circular sets kept** in nested JSON: `circulars` (filtered) and `circulars_all` (full).
- **Slim Excel** = `query` (the input), `actual_output`, `extracted_metadata` only.
- Retriever name match is **case-insensitive** (handles the (detailed) vs (Detailed) casing).
- `KEEP_UNMATCHED_IF_NO_DATES` governs answers citing no date at all.